# Gauthier_Cerema : Test Steger & Frangi

Ce notebook implémente les premiers tests demandés :
1. Téléchargement des données via `gdown`.
2. Téléchargement ciblé du script `steger_gpu.py` depuis GitHub (pas de clonage complet).
3. Filtre Frangi classique (skimage) avec $\sigma=30$.
4. Calcul du Hessien Gaussien (implémentation GPU via PyTorch) avec $\sigma=30$ sur l'image spécifique 'Ombrage 05m base.tif'.

In [ ]:
!pip install rasterio torch torchvision gdown

# 1. Téléchargement du fichier nécessaire depuis GitHub
steger_url = "https://raw.githubusercontent.com/Ludwig-H/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/main/Gauthier_Cerema/steger_gpu.py"
!wget {steger_url} -O steger_gpu.py

graph_url = "https://raw.githubusercontent.com/Ludwig-H/Generalized-Frangi-for-Automatic-Crack-Extraction-on-FIND-dataset/main/Gauthier_Cerema/graph_builder.py"
!wget {graph_url} -O graph_builder.py

# 2. Téléchargement des données (gdown)
# ID du dossier Drive : 1lWi-7oZ064QSh_NjX_xrA5iv03A2hYjQ
!gdown --folder https://drive.google.com/drive/folders/1lWi-7oZ064QSh_NjX_xrA5iv03A2hYjQ -O Gauthier_Cerema_Data

In [ ]:
import sys
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from skimage.filters import frangi
import torch

# Import depuis le fichier téléchargé localement
try:
    from steger_gpu import StegerHessian
    print("Module StegerHessian importé avec succès.")
except ImportError as e:
    print(f"Erreur d'import : {e}. Vérifiez que steger_gpu.py a bien été téléchargé.")

In [ ]:
# --- DEFINITION DU CHEMIN ---
# Les images sont dans le sous-dossier 'Premier_test' du téléchargement
base_data_path = 'Gauthier_Cerema_Data'
folder_path = os.path.join(base_data_path, 'Premier_test')

if os.path.exists(folder_path):
    print(f"Dossier trouvé : {folder_path}")
    files = os.listdir(folder_path)
    tif_files = [f for f in files if f.lower().endswith('.tif')]
    print(f"Fichiers .tif trouvés ({len(tif_files)}) :", tif_files)
    
    # --- Affichage des images brutes ---
    for f in tif_files:
        f_path = os.path.join(folder_path, f)
        try:
            with rasterio.open(f_path) as src:
                img = src.read(1)
                plt.figure(figsize=(6, 6))
                plt.imshow(img, cmap='gray')
                plt.title(f"Image brute : {f}")
                plt.axis('off')
                plt.show()
        except Exception as e:
            print(f"Impossible de lire {f} : {e}")
else:
    print(f"ATTENTION : Le dossier {folder_path} n'a pas été trouvé.")
    if os.path.exists(base_data_path):
        print(f"Contenu de {base_data_path} :", os.listdir(base_data_path))

In [ ]:
# --- 1. FILTRE FRANGI CLASSIQUE (tous les tifs) ---
σ_val = 30

if os.path.exists(folder_path):
    tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif')]

    for file_name in tif_files:
        file_path = os.path.join(folder_path, file_name)
        try:
            with rasterio.open(file_path) as src:
                data = src.read(1)
                
                # Frangi Classique
                print(f"Traitement Frangi (σ={σ_val}) sur {file_name}...")
                response = frangi(data, sigmas=[σ_val])

                plt.figure(figsize=(10, 6))
                plt.imshow(response, cmap='gray')
                plt.colorbar(label='Frangi Response')
                plt.title(f"Frangi (σ={σ_val}) : {file_name}")
                plt.show()
        except Exception as e:
            print(f"Erreur lecture {file_name}: {e}")

In [ ]:
# --- 2. HESSIEN GAUSSIEN PONDÉRÉ (Fusion) ---
weights = {"Ombrage 05m base.tif": 0.5, "MNT 05m.tif": -0.5}
Hxx_acc = None
Hxy_acc = None
Hyy_acc = None
Ix_acc = None
Iy_acc = None
device = 'cuda' if torch.cuda.is_available() else 'cpu'
steger = StegerHessian(σ=σ_val, device=device)

for target_file, weight in weights.items():
    target_path = os.path.join(folder_path, target_file)
    if os.path.exists(target_path):
        print(f"Traitement de {target_file} (poids={weight})...")
        try:
            with rasterio.open(target_path) as src:
                data = src.read(1)
                # Normalisation Image
                data = data.astype(np.float32)
                if data.max() > 0:
                    data = (data - data.min()) / (data.max() - data.min())
                    
                # Conversion PyTorch
                img_tensor = torch.from_numpy(data).unsqueeze(0).unsqueeze(0).to(device)
                
                # Calcul Hessien brut
                ix, iy, ixx, ixy, iyy = steger.compute_hessian(img_tensor)
                λ1, λ2 = steger.compute_eigenvalues(ixx, ixy, iyy)
                
                # Affichage lambda2 individuel (avant normalisation)
                result_np = λ2.squeeze().cpu().numpy()
                plt.figure(figsize=(10, 8))
                plt.imshow(result_np, cmap='RdBu_r')
                plt.colorbar(label='Eigenvalue λ2')
                plt.title(f"Hessian Eigenvalue λ2 (σ={σ_val}) : {target_file}")
                plt.show()

                # Normalisation par max(|λ2|)
                max_val = torch.max(torch.abs(λ2))
                if max_val > 0:
                    ix /= max_val
                    iy /= max_val
                    ixx /= max_val
                    ixy /= max_val
                    iyy /= max_val
                
                # Accumulation pondérée
                if Hxx_acc is None:
                    Ix_acc = weight * ix
                    Iy_acc = weight * iy
                    Hxx_acc = weight * ixx
                    Hxy_acc = weight * ixy
                    Hyy_acc = weight * iyy
                else:
                    Ix_acc += weight * ix
                    Iy_acc += weight * iy
                    Hxx_acc += weight * ixx
                    Hxy_acc += weight * ixy
                    Hyy_acc += weight * iyy
                    
        except Exception as e:
             print(f"Erreur sur {target_file}: {e}")
    else:
        print(f"Fichier cible {target_file} introuvable dans {folder_path}")

# Calcul des valeurs propres du Hessien fusionné
if Hxx_acc is not None:
    print("Calcul des valeurs propres du Hessien fusionné...")
    λ1_fused, λ2_fused = steger.compute_eigenvalues(Hxx_acc, Hxy_acc, Hyy_acc)
    
    result_np = λ2_fused.squeeze().cpu().numpy()
    
    plt.figure(figsize=(12, 10))
    plt.imshow(result_np, cmap='RdBu_r')
    plt.colorbar(label='Eigenvalue λ2 (Fused)')
    plt.title(f"Fused Hessian Eigenvalue λ2 (σ={σ_val})")
    plt.show()
    
    # --- 3. CONSTRUCTION DU GRAPHE SPARSE ET VISUALISATION ---
    # Import local
    import sys
    import os
    import importlib

    # Now that we download graph_builder.py in the first cell, simple import should work
    if os.path.exists('graph_builder.py'):
        import graph_builder as gb
    elif os.path.exists(os.path.join('Gauthier_Cerema', 'graph_builder.py')):
        sys.path.append(os.path.join(os.getcwd(), 'Gauthier_Cerema'))
        import graph_builder as gb
    else:
        # Fallback if file not found locally (maybe running without download cell?)
        try:
            import Gauthier_Cerema.graph_builder as gb
        except ImportError:
            print("Warning: graph_builder.py not found. Ensure the download cell was executed.")
            raise
            
    importlib.reload(gb)
    build_steger_graph = gb.build_steger_graph

    print("Construction du graphe sparse Steger (GPU)...")
    print(f"Plage λ2 fusionné : [{result_np.min():.4f}, {result_np.max():.4f}]")
    nodes, adj = build_steger_graph(Ix_acc, Iy_acc, Hxx_acc, Hxy_acc, Hyy_acc, R=10.0, τ=0.1, dark_ridges=True)
    
    if nodes is not None:
        # Filter: Keep connected nodes
        degrees = np.diff(adj.indptr)
        keep = np.where(degrees > 0)[0]
        
        if len(keep) > 0:
             min_dissims = [adj.data[adj.indptr[i]:adj.indptr[i+1]].min() for i in keep]
             node_response = np.array(min_dissims)
             coords = nodes['coords'][keep]
             directions = nodes['directions'][keep]
        else:
             print("Aucun noeud connecté.")
             node_response = np.array([])
             coords = np.zeros((0,2))
             directions = np.zeros((0,2))
        
        # Visualisation Interactive avec Plotly
        import plotly.graph_objects as go
        import plotly.figure_factory as ff
        
        # 1. Background Image (Heatmap of lambda2)
        # Flip vertically to match image coordinates usually (or rely on layout inversion)
        heatmap = go.Heatmap(z=result_np, colorscale='Gray', zmin=result_np.min(), zmax=result_np.max(), showscale=False, hoverinfo='skip')
        
        # 2. Nodes (Scatter points colored by Min Dissim)
        scatter_nodes = go.Scatter(
            x=coords[:, 0],
            y=coords[:, 1],
            mode='markers',
            marker=dict(
                size=3,
                color=node_response,
                colorscale='Viridis_r',
                colorbar=dict(title="Min Dissim"),
                opacity=0.8
            ),
            name='Steger Nodes',
            text=[f"Dissim: {v:.4f}" for v in node_response],
            hoverinfo='text'
        )

        # 3. Quiver (Arrows) - Subsampled if too many
        step = 1
        if len(coords) > 5000: step = 2
        if len(coords) > 15000: step = 5
        
        sub_x = coords[::step, 0]
        sub_y = coords[::step, 1]
        sub_u = directions[::step, 0]
        sub_v = directions[::step, 1]
        
        # Create Quiver figure (returns figure with data)
        quiver_fig = ff.create_quiver(sub_x, sub_y, sub_u, sub_v,
                                      scale=2.0, 
                                      arrow_scale=0.3,
                                      line=dict(width=1, color='cyan'))
        
        # Combine
        fig = go.Figure(data=[heatmap, scatter_nodes] + list(quiver_fig.data))
        
        fig.update_layout(
            title=f"Interactive Steger Graph (Nodes & Directions) - {len(coords)} nodes",
            width=1600, height=1600,
            yaxis=dict(scaleanchor="x", scaleratio=1, autorange='reversed'), # Image coords top-left origin
            plot_bgcolor='black'
        )
        fig.show()
    else:
        print("Aucun noeud extrait (vérifiez le seuil τ).")